# Visualizações e Análise de Resultados

Criação de gráficos e análise final dos resultados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')

from src.data.load_data import generate_synthetic_data
from src.features.build_features import create_features
from src.models.train_model import ModelTrainer
import yaml

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

df = generate_synthetic_data(n_samples=5000)
df_processed = create_features(df, config, fit=True)

print(f"Shape: {df_processed.shape}")

In [ ]:
target_col = config['model']['target_column']
exclude_cols = [config['features']['id_column'], target_col, 'tenure_group']
feature_cols = [c for c in df_processed.columns if c not in exclude_cols]

X = df_processed[feature_cols]
y = df_processed[target_col]

trainer = ModelTrainer(config)
X_train, X_test, y_train, y_test = trainer.split_data(X, y)
results = trainer.train_all(X_train, y_train, X_test, y_test)
importance = trainer.get_feature_importance()

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_result = results[best_model_name]

churn_rate = df['churn'].mean()
avg_tenure = df['tenure'].mean()
avg_monthly = df['monthly_charges'].mean()

print(f"Churn Rate: {churn_rate:.1%}")
print(f"Tenure Médio: {avg_tenure:.1f} meses")
print(f"Receita Média Mensal: R$ {avg_monthly:.2f}")
print(f"Melhor Modelo: {best_model_name.upper()}")
print(f"ROC AUC: {best_result['roc_auc']:.3f}")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribuição de Churn', 'Percentual'),
    specs=[[{'type': 'bar'}, {'type': 'pie'}]]
)

churn_counts = df['churn'].value_counts()

fig.add_trace(
    go.Bar(x=['Não Churn', 'Churn'], y=churn_counts.values,
           marker_color=['#2ecc71', '#e74c3c']),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=['Não Churn', 'Churn'], values=churn_counts.values,
           marker_colors=['#2ecc71', '#e74c3c'], textinfo='percent+label'),
    row=1, col=2
)

fig.update_layout(title='Análise de Churn', showlegend=False, height=500)
fig.show()

In [ ]:
df['risk_score'] = (
    (df['num_reclamations_30d'] >= 3).astype(int) * 5 +
    (df['num_reclamations_30d'] >= 1).astype(int) * 2 +
    (df['contract_type'] == 'Month-to-month').astype(int) * 3 +
    (df['tenure'] < 6).astype(int) * 2 +
    (df['num_support_tickets'] > 5).astype(int) * 2
)

df['risk_level'] = pd.cut(
    df['risk_score'],
    bins=[-1, 2, 5, 10, 20],
    labels=['Baixo', 'Médio', 'Alto', 'Crítico']
)

colors = {'Baixo': '#2ecc71', 'Médio': '#f39c12', 'Alto': '#e67e22', 'Crítico': '#e74c3c'}

fig = px.histogram(
    df, x='risk_score', color='risk_level',
    category_orders={'risk_level': ['Baixo', 'Médio', 'Alto', 'Crítico']},
    color_discrete_map=colors,
    title='Distribuição do Score de Risco'
)
fig.update_layout(
    xaxis_title='Score de Risco',
    yaxis_title='Número de Clientes',
    barmode='stack'
)
fig.show()

In [ ]:
risk_churn = df.groupby('risk_level', observed=True).agg({
    'churn': ['count', 'sum', 'mean']
}).reset_index()
risk_churn.columns = ['Nível', 'Total', 'Churns', 'Taxa']
risk_churn = risk_churn.sort_values('Taxa', ascending=False)

print("Churn Rate por Nível de Risco:")
print(risk_churn.to_string(index=False))

fig = px.bar(
    risk_churn, x='Nível', y='Taxa', text_auto='.1%',
    color='Taxa', color_continuous_scale='RdYlGn_r'
)
fig.update_layout(title='Churn Rate por Nível de Risco')
fig.show()

In [ ]:
top15 = importance.head(15).sort_values('importance', ascending=True)

fig = px.bar(
    top15, y='feature', x='importance', orientation='h',
    text_auto='.4f', color='importance', color_continuous_scale='Viridis'
)
fig.update_layout(
    title=f'Top 15 Features Mais Importantes - {best_model_name.upper()}',
    xaxis_title='Importância',
    yaxis_title='Feature',
    showlegend=False
)
fig.show()

In [ ]:
models = list(results.keys())

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Métricas de Classificação', 'ROC AUC')
)

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    values = [results[m].get(metric, 0) for m in models]
    fig.add_trace(
        go.Bar(name=metric, x=models, y=values, texttemplate='%{y:.3f}'),
        row=1, col=1
    )

auc_values = [results[m].get('roc_auc', 0) for m in models]
fig.add_trace(
    go.Bar(name='ROC AUC', x=models, y=auc_values,
           texttemplate='%{y:.3f}', marker_color='#3498db'),
    row=1, col=2
)

fig.update_layout(title='Comparação de Modelos', barmode='group', height=500)
fig.show()

In [ ]:
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
if 'customer_id' in numeric_cols:
    numeric_cols.remove('customer_id')

corr = df_processed[numeric_cols].corr()

fig = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='RdBu_r',
    title='Correlações entre Features',
    width=800, height=800
)
fig.show()

In [ ]:
print("=" * 60)
print("CONCLUSÕES DA ANÁLISE")
print("=" * 60)

print("""
1. FATORES PRINCIPAIS DE CHURN:
   - Reclamações: clientes com 3+ reclamações em 30 dias
   - Contrato MTM: contratos month-to-month
   - Baixa tenure: clientes < 6 meses

2. SEGMENTAÇÃO DE RISCO:
   - Baixo Risco (0-2): taxa de churn baixa
   - Médio Risco (3-5): atenção necessária
   - Alto Risco (6-10): intervenção recomendada
   - Crítico (11+): ação imediata

3. MODELO PREDITIVO:
   - Melhor modelo: {}
   - ROC AUC: {:.3f}
   - Features mais importantes: num_reclamations_30d, tenure, contract_type

4. RECOMENDAÇÕES:
   - Sistema de alerta para 3+ reclamações
   - Foco em contratos MTM com baixa tenure
   - Programa de engajamento para clientes críticos
""".format(best_model_name.upper(), best_result['roc_auc']))